In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS']='0'

In [2]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')


In [3]:
!pip install tensorflow

In [4]:
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [5]:
tf.get_logger().setLevel('ERROR')

In [6]:
print('=' * 60)
print('STEP 03 - MODEL CREATION')
print('=' * 60)

STEP 03 - MODEL CREATION


In [7]:
#load cleaned Data (from step 02)
df_train= pd.read_csv(r"C:\Users\FLAKES\OneDrive\Documents\last tem\Computer system project\store\raw\train_grocery_inventory.xls", parse_dates=['Date'])
df_test= pd.read_csv(r"C:\Users\FLAKES\OneDrive\Documents\last tem\Computer system project\store\raw\test_grocery_inventory.xls", parse_dates=['Date'])

In [8]:
FEATURE_COLS = [
    'Inventory Level', 'Units Ordered', 'Demand Forecast',
    'Price', 'Discount', 'Competitor Pricing',
    'Weather_Enc', 'Season_Enc', 'Holiday/Promotion',
    'Region_North', 'Region_South', 'Region_West',
    'DayOfWeek', 'DayOfMonth', 'Month', 'WeekOfYear', 'IsWeekend',
    'Units_Sold_Lag1', 'Units_Sold_Lag3', 'Units_Sold_Lag7', 'Units_Sold_Lag14',
    'Inventory_Lag1', 'Inventory_Lag3', 'Inventory_Lag7', 'Inventory_Lag14',
    'Rolling_Mean_7d', 'Rolling_Std_7d', 'Rolling_Mean_14d'
]
TARGET_COL = 'Units Sold'

X_train = df_train[FEATURE_COLS].values
y_train = df_train[TARGET_COL].values
X_test  = df_test[FEATURE_COLS].values
y_test  = df_test[TARGET_COL].values

print(f"Data loaded")
print(f"  Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"  Features : {len(FEATURE_COLS)}  |  Target : {TARGET_COL} (scaled 0–1)")

Data loaded
  Train : (10568, 28)  |  Test : (2643, 28)
  Features : 28  |  Target : Units Sold (scaled 0–1)


In [9]:
#lstm
print('\n' + '-' * 60)
print('LSTM MODEL (TensorFlow / Keras)')
print('-' * 60)


------------------------------------------------------------
LSTM MODEL (TensorFlow / Keras)
------------------------------------------------------------


In [10]:
SEQ_LEN = 14

def make_sequences(X, y, seq_len):
    Xs, ys=[],[]
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = make_sequences(X_train, y_train, SEQ_LEN)
X_test_seq, y_test_seq = make_sequences(X_test, y_test, SEQ_LEN)

print(f'Sequence built - window = {SEQ_LEN} days')
print(f'LSTM train shape : {X_train_seq.shape} (samples, timesteps, features)')
print(f'LSTM test shape : {X_test_seq.shape}')

Sequence built - window = 14 days
LSTM train shape : (10554, 14, 28) (samples, timesteps, features)
LSTM test shape : (2629, 14, 28)


In [11]:
#build LSTM architecture
n_features = X_train_seq.shape[2]

lstm_model = Sequential([
    LSTM(64, return_sequences = True, input_shape=(SEQ_LEN,n_features)),
    Dropout(0.2),
    LSTM(32, return_sequences = False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
], name= 'LSTM_StockLevel')

lstm_model.compile (optimizer= 'adam', loss='mse', metrics=['mae'])

print(f'\n LSTM Architecture')
lstm_model.summary(print_fn=lambda x: print(f'{x}'))


 LSTM Architecture


Model: "LSTM_StockLevel"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 14, 64)              │          23,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 14, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼───────────────

In [12]:
#callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5,
                          restore_best_weights= True, verbose=0)

checkpoint= ModelCheckpoint('lstm_best_model.keras', monitor='val_loss',
                          save_best_only=True, verbose=0)

print(f'\n Training LSTM (epochs=50, batch= 32, early stoping patient=5)')
history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_split=0.15,
    callbacks= [early_stop, checkpoint],
    verbose=0
)

history_df.to_csv(
    r"C:\Users\FLAKES\OneDrive\Documents\last tem\Computer system project\store\lstm_training_history.csv",
    index=False
)

print("✔ LSTM training history saved to lstm_training_history.csv")

epochs_run = len(history.history['loss'])
best_val = min (history.history['val_loss'])
print(f" Training complete {epochs_run} epochs run | Best val_loss = {best_val:.5f}")


 Training LSTM (epochs=50, batch= 32, early stoping patient=5)


NameError: name 'history_df' is not defined

In [ ]:
#SVR
svr_model = SVR(
    kernel='rbf',
    C=100,
    epsilon= 0.01,
    gamma= 'scale'
)

print(f'SVR Hyperparameters:')
print(f' kernel = rbf (nonlinear)')
print(f' C = 100 (regularization)')
print(f' epsilon = 0.01 (tolerance tube)')
print(f' gamma = scale (auto)')

print(f'\n Training SVR on {X_train.shape[0]:,} samples ..')
svr_model.fit(X_train, y_train)
print(f'SVR training complete')

In [ ]:
#Save SVR model
with open ('svr_model.pkl', 'wb') as f:
    pickle.dump(svr_model, f)
print(f'SVR model an')